In [53]:
# ============================================================================
#  DATA LOADING
# ============================================================================

import random
import time
import uuid
import math
import numpy as np
import io
from collections import OrderedDict
from PIL import Image
from torchvision import transforms
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode
from torch.amp import GradScaler, autocast
from torch.utils.checkpoint import checkpoint
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
from torch.utils.data import Dataset, DataLoader

import os
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import efficientnet_b1, EfficientNet_B1_Weights

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

class RandomJpegCompression:
    def __init__(self, p=0.35, quality_min=65, quality_max=95):
        self.p, self.quality_min, self.quality_max = p, quality_min, quality_max

    def __call__(self, img):
        if random.random() > self.p:
            return img
        q = random.randint(self.quality_min, self.quality_max)
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=q, subsampling=0)
        buf.seek(0)
        return Image.open(buf).convert("RGB")


class RandomGaussianBlurX:
    def __init__(self, kmin=13, kmax=20, smin=1.2, smax=2.3):
        self.kmin, self.kmax = kmin, kmax
        self.smin, self.smax = smin, smax

    def __call__(self, img):
        k_lo = self.kmin if self.kmin % 2 == 1 else self.kmin + 1
        k_hi = self.kmax if self.kmax % 2 == 1 else self.kmax - 1
        if k_hi < k_lo:
            k_hi = k_lo
        k = random.randrange(k_lo, k_hi + 1, 2)
        sigma = random.uniform(self.smin, self.smax)
        return TF.gaussian_blur(img, kernel_size=[k, k], sigma=[sigma, sigma])


normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])

# transform_x = transforms.Compose([
#     RandomGaussianBlurX(),
#     transforms.ToTensor(),
#     normalize,
# ])

transform_clean = transforms.Compose([
    transforms.ToTensor(),
    normalize,
])


def denormalize(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406], device=tensor.device).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=tensor.device).view(3, 1, 1)
    return torch.clamp(tensor * std + mean, 0, 1)


def tensor_to_image(tensor):
    return Image.fromarray((tensor.permute(1, 2, 0).cpu().numpy() * 255).astype('uint8'))


def random_right_angle_rotate_pair(x_img, y_img, p=0.5):
    if random.random() > p:
        return x_img, y_img
    angle = random.choice([90, 180, 270])
    ops = {90: Image.Transpose.ROTATE_90, 180: Image.Transpose.ROTATE_180,
           270: Image.Transpose.ROTATE_270}
    return x_img.transpose(ops[angle]), y_img.transpose(ops[angle])


def apply_augmentation(x_img, y_img):
    if random.random() > 0.5:
        x_img, y_img = TF.hflip(x_img), TF.hflip(y_img)
    if random.random() > 0.5:
        x_img, y_img = TF.vflip(x_img), TF.vflip(y_img)
    x_img, y_img = random_right_angle_rotate_pair(x_img, y_img, p=0.5)
    if random.random() < 0.35:
        x_img = TF.adjust_brightness(x_img, random.uniform(0.96, 1))
        x_img = TF.adjust_contrast(x_img, random.uniform(0.96, 1))
        x_img = TF.adjust_saturation(x_img, random.uniform(0.96, 1))
    x_img = RandomJpegCompression()(x_img)
    return x_img, y_img


def get_filenames_from_y(y_folder):
    exts = (".png", ".jpg", ".jpeg", ".webp")
    filenames = sorted(f for f in os.listdir(y_folder) if f.lower().endswith(exts))
    random.shuffle(filenames)
    return filenames


def split_filenames(filenames, train_ratio=0.96):
    n = int(len(filenames) * train_ratio)
    return filenames[:n], filenames[n:]

import torch.nn.functional as F

class SuperResDataset(Dataset):
    def __init__(self, y_folder, filenames):
        self.y_folder = y_folder
        self.filenames = filenames

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        # The CPU ONLY reads the file and converts it to a raw, normalized tensor.
        filename = self.filenames[idx]
        img = Image.open(os.path.join(self.y_folder, filename)).convert("RGB")
        
        # transform_clean = ToTensor() + Normalize()
        return transform_clean(img)

class DynamicGaussianBlur:
    """Scales the blur kernel automatically based on the image size."""
    def __init__(self, kernel_ratio=0.002, smin=0.4, smax=1.2):
        self.kernel_ratio = kernel_ratio # Kernel will be 3% of image width
        self.smin, self.smax = smin, smax

    def __call__(self, img):
        # Calculate dynamic kernel size
        w, h = img.size
        k = int(min(w, h) * self.kernel_ratio)
        k = k if k % 2 == 1 else k + 1 # Ensure odd number
        k = max(3, k) # Never go below a 3x3 kernel
        
        sigma = random.uniform(self.smin, self.smax)
        return TF.gaussian_blur(img, kernel_size=[k, k], sigma=[sigma, sigma])

transform_x = transforms.Compose([
    DynamicGaussianBlur(kernel_ratio=0.03), # Safe at any resolution
    transforms.ToTensor(),
    normalize,
])

def load_pretrain_images(y_folder, filenames, batch_size, x_size=64, y_size=128, augment=False):
    """
    PROPER PRETRAIN LOADER:
    Extracts a native y_size crop from the full image, then downscales it to x_size.
    Preserves actual textures without scale-warping.
    """
    x_images, y_images = [], []
    for filename in filenames:
        img = Image.open(os.path.join(y_folder, filename)).convert("RGB")
        
        # 1. THE FIX: Crop a true high-res patch from the original image
        # This gives you a native 128x128 piece of the actual 4K texture
        i, j, h, w = transforms.RandomCrop.get_params(img, output_size=(y_size, y_size))
        y_img = TF.crop(img, i, j, h, w)
        
        # 2. Downscale the high-res crop to create the degraded input
        x_img = TF.resize(y_img, (x_size, x_size), interpolation=InterpolationMode.BICUBIC)

        if augment:
            # We skip JPEG compression here if the image is too small (e.g., < 128)
            # to avoid destroying the 8x8 block grids.
            if x_size >= 128:
                x_img = RandomJpegCompression()(x_img)
            
            x_img, y_img = apply_augmentation(x_img, y_img)

        # Dynamic Blur is handled inside transform_x now
        x_tensor = transform_x(x_img)
        y_tensor = transform_clean(y_img)

        if augment and random.random() < 0.35:
            # Scale the noise down slightly for smaller images to avoid burying the signal
            noise_scale = 0.01 if x_size <= 64 else 0.02
            x_tensor = x_tensor + torch.randn_like(x_tensor) * random.uniform(0.0, noise_scale)

        x_images.append(x_tensor)
        y_images.append(y_tensor)

        if len(x_images) == batch_size:
            yield torch.stack(x_images).cuda(), torch.stack(y_images).cuda()
            x_images, y_images = [], []
            
    if x_images:
        yield torch.stack(x_images).cuda(), torch.stack(y_images).cuda()
        
        
        
def load_images_from_filenames(y_folder, filenames, batch_size,
                                x_size=512, y_size=1024, augment=False):
    """Full training loader. X = degraded at x_size, Y = clean at y_size."""
    x_images, y_images = [], []
    for filename in filenames:
        y_img = Image.open(os.path.join(y_folder, filename)).convert("RGB")
        # y_img = TF.resize(y_img, (y_size, y_size), interpolation=InterpolationMode.BICUBIC)
        x_img = TF.resize(y_img, (x_size, x_size), interpolation=InterpolationMode.LANCZOS)

        if augment:
            x_img, y_img = apply_augmentation(x_img, y_img)

        x_tensor = transform_x(x_img)
        y_tensor = transform_clean(y_img)

        if augment and random.random() < 0.35:
            x_tensor = x_tensor + torch.randn_like(x_tensor) * random.uniform(0.0, 0.02)

        x_images.append(x_tensor)
        y_images.append(y_tensor)

        if len(x_images) == batch_size:
            yield torch.stack(x_images).cuda(), torch.stack(y_images).cuda()
            x_images, y_images = [], []
    if x_images:
        yield torch.stack(x_images).cuda(), torch.stack(y_images).cuda()


def load_pretrain_images(y_folder, filenames, batch_size, x_size=64, y_size=128, augment=False):
    """
    PROPER PRETRAIN LOADER:
    Extracts a native y_size crop from the full image, then downscales it to x_size.
    Preserves actual textures without scale-warping.
    """
    x_images, y_images = [], []
    for filename in filenames:
        img = Image.open(os.path.join(y_folder, filename)).convert("RGB")
        
        # 1. THE FIX: Crop a true high-res patch from the original image
        # This gives you a native 128x128 piece of the actual 4K texture
        i, j, h, w = transforms.RandomCrop.get_params(img, output_size=(y_size, y_size))
        y_img = TF.crop(img, i, j, h, w)
        
        # 2. Downscale the high-res crop to create the degraded input
        x_img = TF.resize(y_img, (x_size, x_size), interpolation=InterpolationMode.BICUBIC)

        if augment:
            # We skip JPEG compression here if the image is too small (e.g., < 128)
            # to avoid destroying the 8x8 block grids.
            if x_size >= 128:
                x_img = RandomJpegCompression()(x_img)
            
            x_img, y_img = apply_augmentation(x_img, y_img)

        # Dynamic Blur is handled inside transform_x now
        x_tensor = transform_x(x_img)
        y_tensor = transform_clean(y_img)

        if augment and random.random() < 0.35:
            # Scale the noise down slightly for smaller images to avoid burying the signal
            noise_scale = 0.01 if x_size <= 64 else 0.02
            x_tensor = x_tensor + torch.randn_like(x_tensor) * random.uniform(0.0, noise_scale)

        x_images.append(x_tensor)
        y_images.append(y_tensor)

        if len(x_images) == batch_size:
            yield torch.stack(x_images).cuda(), torch.stack(y_images).cuda()
            x_images, y_images = [], []
            
    if x_images:
        yield torch.stack(x_images).cuda(), torch.stack(y_images).cuda()

C:\Users\ooo\AppData\Local\Temp\ipykernel_5008\405668685.py:18: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [2]:

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = "expandable_segments:True"
gc.collect()


class Swish(nn.Module):
    def forward(self, x):
        return x * torch.sigmoid(x)

class SqueezeExcitation(nn.Module):
    def __init__(self, in_ch, se_ratio=0.25):
        super().__init__()
        reduced_ch = max(1, int(in_ch * se_ratio))
        self.fc1 = nn.Conv2d(in_ch, reduced_ch, kernel_size=1)
        self.fc2 = nn.Conv2d(reduced_ch, in_ch, kernel_size=1)
        self.activation = Swish()

    def forward(self, x):
        se = x.mean((2, 3), keepdim=True)
        se = self.activation(self.fc1(se))
        se = torch.sigmoid(self.fc2(se))
        return x * se

class PixelShuffle_ICNR(nn.Module):
    def __init__(self, ni, nf=None, scale=2, blur=True):
        super().__init__()
        nf = nf or ni
        self.scale = scale
        
        # 1. The convolution that expands channels
        self.conv = nn.Conv2d(ni, nf * (scale ** 2), kernel_size=1)
        
        # 2. Apply mathematical ICNR Initialization
        self._icnr_init()
        
        # 3. Shuffle into spatial dimensions
        self.shuf = nn.PixelShuffle(scale)
        
        # 4. Low-pass filter to physically blend the sub-pixels
        self.blur = nn.Sequential(
            nn.ReplicationPad2d(1),
            nn.AvgPool2d(3, stride=1)
        ) if blur else None
        
        self.relu = nn.LeakyReLU(0.1, inplace=True)

    def _icnr_init(self):
        """Overrides default random weights to prevent checkerboard artifacts."""
        with torch.no_grad():
            out_c, in_c, h, w = self.conv.weight.shape
            base_c = out_c // (self.scale ** 2)
            
            # Generate Kaiming random weights for the base channels
            base_weight = torch.empty(base_c, in_c, h, w)
            nn.init.kaiming_normal_(base_weight)
            
            # Duplicate the exact weights for the sub-pixels
            repeated_weight = base_weight.repeat_interleave(self.scale ** 2, dim=0)
            self.conv.weight.data.copy_(repeated_weight)
            
            if self.conv.bias is not None:
                nn.init.zeros_(self.conv.bias)

    def forward(self, x):
        x = self.conv(x)
        x = self.relu(x)
        x = self.shuf(x)
        if self.blur:
            x = self.blur(x)
        return x

def GN(c, max_groups=32):
    g = min(max_groups, c)
    while c % g != 0 and g > 1:
        g -= 1
    return nn.GroupNorm(g, c)


# ============================================================================
#  DEPTHWISE SEPARABLE CONV (8x cheaper than standard 3x3 conv)
# ============================================================================

class DepthwiseSeparableConv(nn.Module):
    #Depthwise separable convolution: depthwise 3x3 + pointwise 1x1.
    def __init__(self, channels):
        super().__init__()
        self.depthwise = nn.Conv2d(channels, channels, kernel_size=3, padding=1,
                                   padding_mode='reflect',
                                   groups=channels, bias=False)
        self.pointwise = nn.Conv2d(channels, channels, kernel_size=1, bias=False)
        self.norm = GN(channels)
        self.act = nn.LeakyReLU(0.1, inplace=True)

    def forward(self, x):
        return self.act(self.norm(self.pointwise(self.depthwise(x))))


# ============================================================================
#  SHARED REFINEMENT BLOCK (DSA + DepthwiseSepConv, applied N times)
# ============================================================================

class SharedRefineBlock(nn.Module):
    """One DSA attention + one depthwise separable conv, designed to be
    called in a loop N times with shared weights. Each call refines the
    tensor further. Uses gradient checkpointing internally."""
    def __init__(self, channels, num_heads, top_k=64):
        super().__init__()
        self.attn = DeepSeekSparseAttention2D(channels, num_main_heads=num_heads, top_k=top_k)
        self.conv = DepthwiseSeparableConv(channels)
        # Residual gate so the conv branch starts safe
        self.conv_gate = nn.Parameter(torch.tensor(0.5))

    def forward(self, x):
        x = self.attn(x)                          # DSA already does x + gamma*attn
        x = x + self.conv_gate * self.conv(x)      # gated depthwise sep conv residual
        return x


class EfficientNetB1SingleEncoder(nn.Module):
    def __init__(self, weights=EfficientNet_B1_Weights.DEFAULT):
        super().__init__()
        base_model = efficientnet_b1(weights=weights)
        self.features = base_model.features[:-1]

    def forward(self, x):
        chosen_indices = [1, 2, 3, 5, 7]
        temp_feats = []
        for idx, layer in enumerate(self.features):
            x = layer(x)
            if idx in chosen_indices:
                temp_feats.append(x)
        feats = [temp_feats[4], temp_feats[3], temp_feats[2],
                 temp_feats[1], temp_feats[0]]
        return x, feats

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, padding_mode='reflect', bias=False)

    def forward(self, x):
        x_dtype = x.dtype
        with torch.autocast(device_type='cuda', enabled=False):
            avg_out = x.mean(dim=1, keepdim=True).float()
            max_out = x.amax(dim=1, keepdim=True).float()
            concat = torch.cat([avg_out, max_out], dim=1)
            attn = torch.sigmoid(self.conv(concat))
        return x * attn.to(dtype=x_dtype)


def run_module_fp32(module, x):
    x_dtype = x.dtype
    with torch.autocast(device_type='cuda', enabled=False):
        y = module(x.float())
    return y.to(dtype=x_dtype)


class RefinementBlock(nn.Module):
    def __init__(self, in_ch, mid_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, mid_ch, kernel_size=3, padding=1, padding_mode='reflect')
        self.relu1 = nn.LeakyReLU(0.1, inplace=True)
        self.conv2 = nn.Conv2d(mid_ch, mid_ch, kernel_size=3, padding=1, padding_mode='reflect')
        self.relu2 = nn.LeakyReLU(0.1, inplace=True)
        self.conv3 = nn.Conv2d(mid_ch, out_ch, kernel_size=3, padding=1, padding_mode='reflect')
        
        # THE FIX: Check if channels match. 
        self.use_projection = (in_ch != out_ch)
        
        # Only build the 1x1 Conv if we actually need to change the tensor shape
        if self.use_projection:
            self.skip_conv = nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)

    def forward(self, x):
        # If shapes don't match, use the 1x1 projection. 
        # If they DO match, act as a pure, free identity highway.
        skip = self.skip_conv(x) if self.use_projection else x
        
        out = self.relu1(self.conv1(x))
        out = self.relu2(self.conv2(out))
        out = self.conv3(out)
        
        return out + skip


# ============================================================================
#  mHC CORE
# ============================================================================

class RMSNorm2d(nn.Module):
    """RMSNorm for (B, C, H, W) feature maps. Paper specifies RMSNorm, not LayerNorm."""
    def __init__(self, dim, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(1, dim, 1, 1))

    def forward(self, x):
        x_float = x.float()
        rms = torch.sqrt(x_float.pow(2).mean(dim=1, keepdim=True) + self.eps)
        return (x_float / rms).to(x.dtype) * self.scale


def sinkhorn_knopp(mat, num_iters=10):
    """
    Project onto the Birkhoff polytope (doubly stochastic matrices).
    mat: (..., n, n) raw logits → exp → iterative row/col normalization.
    """
    m = torch.exp(mat.float())
    for _ in range(num_iters):
        m = m / (m.sum(dim=-1, keepdim=True) + 1e-8)
        m = m / (m.sum(dim=-2, keepdim=True) + 1e-8)
    return m.to(mat.dtype)


class mHCUnit(nn.Module):
    """
    Core mHC mechanism wrapping a single shape-preserving layer.

    Given widened stream (B, n*C, H, W):
      1. H_pre aggregates n streams → layer input (B, C, H, W)
      2. layer_fn processes it
      3. H_post broadcasts output back to n streams
      4. H_res mixes the residual streams (doubly stochastic)
      5. New stream = H_res @ old_stream + H_post * layer_output
    """
    def __init__(self, dim, expansion=4, sinkhorn_iters=5):
        super().__init__()

        # dim = number of channels per stream.  Example: dim=64
        self.dim = dim

        # n = number of parallel streams.  Example: n=4
        # So the total "widened" tensor is n*dim = 256 channels.
        self.n = expansion

        # stream_dim = total channels in the widened stream = n * dim = 256
        self.stream_dim = dim * expansion

        # How many times to iterate Sinkhorn-Knopp (more = closer to true
        # doubly stochastic, 10 is plenty for a 4x4 matrix).
        self.sinkhorn_iters = sinkhorn_iters

        # =====================================================================
        # STATIC BIASES — fixed starting values for the three coefficient sets.
        # These define the "default" behavior before the network learns anything.
        # They are nn.Parameters so backprop can adjust them during training.
        # =====================================================================

        # b_pre: shape (n,) = (4,) — one bias value per stream.
        #
        # These biases feed into softmax to produce the 4 weights that collapse
        # the 4 streams into a single layer input. We want stream[0] (the main
        # signal) to dominate at the start, so we set its bias high (2.0) and
        # the rest low (-1.0).
        #
        # softmax([2.0, -1.0, -1.0, -1.0]) ≈ [0.95, 0.017, 0.017, 0.017]
        # → at init, the layer input is ~95% stream[0], barely any from others.
        b_pre_init = torch.full((expansion,), -1.0)
        b_pre_init[0] = 2.0
        self.b_pre = nn.Parameter(b_pre_init.clone())

        # b_post: shape (n,) = (4,) — one bias value per stream.
        #
        # These biases feed into sigmoid to produce per-stream gates that
        # control how much of the layer's output each stream receives.
        # Same init values, but sigmoid interprets them differently:
        #   sigmoid(2.0) = 0.88  → stream[0] receives 88% of the layer output
        #   sigmoid(-1.0) = 0.27 → streams[1-3] receive 27% each
        self.b_post = nn.Parameter(b_pre_init.clone())

        # b_res: shape (n, n) = (4, 4) — a full mixing matrix.
        #
        # WHY 4×4? Because this matrix defines how each of the 4 NEW streams
        # is built from the 4 OLD streams:
        #   new_stream[0] = m[0,0]*old[0] + m[0,1]*old[1] + m[0,2]*old[2] + m[0,3]*old[3]
        #   new_stream[1] = m[1,0]*old[0] + m[1,1]*old[1] + m[1,2]*old[2] + m[1,3]*old[3]
        #   ...
        # Row i defines where new_stream[i] comes from. Column j defines how
        # much old_stream[j] contributes to all new streams.
        #
        # We want this to start as near-identity (each stream keeps itself):
        #   [[ 5, -3, -3, -3],     after Sinkhorn     [[~0.97, ~0.01, ~0.01, ~0.01],
        #    [-3,  5, -3, -3],     ──────────────→      [~0.01, ~0.97, ~0.01, ~0.01],
        #    [-3, -3,  5, -3],                          [~0.01, ~0.01, ~0.97, ~0.01],
        #    [-3, -3, -3,  5]]                          [~0.01, ~0.01, ~0.01, ~0.97]]
        b_res_init = torch.full((expansion, expansion), -3.0)
        b_res_init.fill_diagonal_(5.0)
        self.b_res = nn.Parameter(b_res_init)

        # =====================================================================
        # GATING SCALARS (alpha) — control how much the dynamic (data-dependent)
        # part contributes vs the static biases above.
        #
        # At init, alpha = 0.01, so the dynamic part is multiplied by 0.01,
        # making it negligible. The static biases above dominate completely.
        # As training progresses, alpha grows and the network starts using
        # data-dependent coefficients.
        # =====================================================================
        self.alpha_pre = nn.Parameter(torch.tensor(0.01))
        self.alpha_post = nn.Parameter(torch.tensor(0.01))
        self.alpha_res = nn.Parameter(torch.tensor(0.01))

        # =====================================================================
        # DYNAMIC PROJECTIONS (phi) — 1x1 convolutions that look at the current
        # stream content and produce per-pixel coefficients.
        #
        # Think of these as: "look at what's in the streams RIGHT NOW and decide
        # how to mix them for THIS specific pixel." Different pixels can get
        # different mixing coefficients.
        #
        # phi_pre:  input = 256ch stream → output = 4 values (one weight per stream)
        # phi_post: input = 256ch stream → output = 4 values (one gate per stream)
        # phi_res:  input = 256ch stream → output = 16 values (a full 4×4 matrix)
        #
        # phi_res outputs 4*4=16 because it needs to produce a complete mixing
        # matrix. phi_pre and phi_post only output 4 because they are vectors,
        # not matrices.
        # =====================================================================
        self.phi_pre = nn.Conv2d(self.stream_dim, expansion, 1, bias=False)
        self.phi_post = nn.Conv2d(self.stream_dim, expansion, 1, bias=False)
        self.phi_res = nn.Conv2d(self.stream_dim, expansion * expansion, 1, bias=False)

        # RMSNorm on the full 256ch stream — normalizes the stream before
        # feeding it into the phi convolutions, for training stability.
        self.norm = RMSNorm2d(self.stream_dim)

        # Initialize all phi convolution weights to ZERO.
        # This means at the start of training:
        #   phi_pre(stream) = 0, phi_post(stream) = 0, phi_res(stream) = 0
        # So the raw coefficients = alpha * 0 + static_bias = just the static bias.
        # The network starts with pure static behavior (identity-like) and
        # gradually learns to use the dynamic data-dependent part.
        nn.init.zeros_(self.phi_pre.weight)
        nn.init.zeros_(self.phi_post.weight)
        nn.init.zeros_(self.phi_res.weight)
        
#         self.output_gate = nn.Parameter(torch.zeros(1))
        self.output_gate = nn.Parameter(torch.tensor(0.5))

    def forward(self, stream, layer_fn):
        """
        stream: (B, n*C, H, W) — the widened multi-stream tensor
        layer_fn: callable (B, C, H, W) → (B, C, H, W) — e.g. a Conv3x3 block
        Returns: (B, n*C, H, W) — updated multi-stream tensor
        """
        B, nC, H, W = stream.shape
        C, n = self.dim, self.n
        # Example: B=2, C=64, n=4, nC=256, H=W=44

        # =====================================================================
        # STEP 0: Normalize the stream, then compute the three sets of
        # coefficients. Each set has a STATIC part (the bias, same for every
        # pixel) and a DYNAMIC part (produced by a 1x1 conv, different per pixel).
        # The formula for each is:  raw = alpha * dynamic + bias
        # =====================================================================

        # Normalize the 256ch stream for stable coefficient computation
        s_norm = self.norm(stream)

        # Run the three 1x1 convolutions on the normalized stream.
        # These look at what's currently in ALL 4 streams and produce
        # data-dependent coefficients for this specific pixel.
        #
        # phi_pre:  256ch → 4 values per pixel (one weight per stream)
        h_pre_dyn = self.phi_pre(s_norm)                         # (B, 4, H, W)
        # phi_post: 256ch → 4 values per pixel (one gate per stream)
        h_post_dyn = self.phi_post(s_norm)                       # (B, 4, H, W)
        # phi_res:  256ch → 16 values per pixel → reshaped into a 4×4 matrix
        h_res_dyn = self.phi_res(s_norm).view(B, n, n, H, W)    # (B, 4, 4, H, W)

        # Combine: raw = (tiny alpha) * (data-dependent) + (static bias)
        # At init: alpha=0.01, phi outputs 0 → raw = 0 + bias = just the bias.
        # Over training: alpha grows, phi learns → raw becomes data-dependent.
        #
        # b_pre is shape (4,), we reshape to (1, 4, 1, 1) so it broadcasts
        # across batch, height, and width dimensions.
        h_pre_raw = self.alpha_pre * h_pre_dyn + self.b_pre.view(1, n, 1, 1)
        h_post_raw = self.alpha_post * h_post_dyn + self.b_post.view(1, n, 1, 1)
        # b_res is shape (4, 4), reshaped to (1, 4, 4, 1, 1) for broadcasting.
        h_res_raw = self.alpha_res * h_res_dyn + self.b_res.view(1, n, n, 1, 1)

        # =====================================================================
        # STEP 1: Apply manifold constraints to turn raw numbers into valid
        # coefficients that can't cause signal explosion.
        # =====================================================================

        # H_pre: softmax across the n=4 streams → weights that sum to 1.
        # At init: softmax([2, -1, -1, -1]) ≈ [0.95, 0.017, 0.017, 0.017]
        # This means the layer input will be ~95% stream[0].
        H_pre = torch.softmax(h_pre_raw, dim=1)  # (B, 4, H, W)

        # H_post: sigmoid on each value independently → each gate ∈ (0, 1).
#         At init: sigmoid(2)=0.88 for stream[0], sigmoid(-1)=0.27 for others.
        H_post = torch.sigmoid(h_post_raw)        # (B, 4, H, W)

        # H_res: Sinkhorn-Knopp projection → doubly stochastic 4×4 matrix.
        # We need to reshape so each pixel's 4×4 matrix is in the last two dims.
        # (B, 4, 4, H, W) → permute → (B, H, W, 4, 4) → flatten → (B*H*W, 4, 4)
        h_sk = h_res_raw.permute(0, 3, 4, 1, 2).reshape(-1, n, n)  # (B*H*W, 4, 4)
        # Run Sinkhorn: exp() then alternate row/column normalization 10 times.
        # Result: every row sums to 1, every column sums to 1 → can't explode.
        H_res = sinkhorn_knopp(h_sk, self.sinkhorn_iters)
        # Reshape back: (B*H*W, 4, 4) → (B, H, W, 4, 4) → permute → (B, 4, 4, H, W)
        H_res = H_res.view(B, H, W, n, n).permute(0, 3, 4, 1, 2)  # (B, 4, 4, H, W)

        # =====================================================================
        # STEP 2: Use the coefficients to process the streams.
        # =====================================================================

        # Reshape stream from (B, 256, H, W) into (B, 4, 64, H, W)
        # so we can work with individual streams.
        stream_r = stream.view(B, n, C, H, W)

        # --- H_pre: collapse 4 streams into 1 layer input ---
        #
        # H_pre is (B, 4, H, W) — 4 weights per pixel.
        # stream_r is (B, 4, 64, H, W) — 4 streams of 64 channels.
        #
        # unsqueeze(2) adds a channel dim: H_pre becomes (B, 4, 1, H, W)
        # Multiply: (B, 4, 1, H, W) * (B, 4, 64, H, W) = (B, 4, 64, H, W)
        #   → each stream's 64 channels are scaled by that stream's weight
        # .sum(dim=1): add up across the 4 streams → (B, 64, H, W)
        #
        # This is a weighted average: input = w0*stream[0] + w1*stream[1] + ...
        layer_input = (H_pre.unsqueeze(2) * stream_r).sum(dim=1)  # (B, 64, H, W)

        # --- Run the actual layer (e.g. Conv3x3 + GroupNorm + LeakyReLU) ---
        # Only ONE 64-channel tensor goes in, ONE 64-channel tensor comes out.
        layer_output = layer_fn(layer_input)  # (B, 64, H, W)

        # --- H_post: distribute layer output to all 4 streams ---
        #
        # H_post is (B, 4, H, W) — one gate per stream.
        # layer_output is (B, 64, H, W).
        #
        # unsqueeze(2): H_post becomes (B, 4, 1, H, W)
        # unsqueeze(1): layer_output becomes (B, 1, 64, H, W)
        # Multiply: (B, 4, 1, H, W) * (B, 1, 64, H, W) = (B, 4, 64, H, W)
        #   → the same layer output is copied to all 4 streams, each scaled
        #     by its own gate value (0.88 for stream[0], 0.27 for others at init)
        stream_update = H_post.unsqueeze(2) * layer_output.unsqueeze(1)  # (B, 4, 64, H, W)

        # --- H_res: mix old streams via the 4×4 doubly stochastic matrix ---
        #
        # For each pixel, this does a matrix multiply:
        #   mixed_stream[i] = Σ_j  H_res[i,j] * old_stream[j]
        #
        # einsum letters: b=batch, i=output_stream, j=input_stream,
        #                 c=channel, h=height, w=width
        # j appears in both inputs but not output → summed over (matrix multiply)
        #
        # At init H_res ≈ identity, so mixed ≈ old streams unchanged.
        mixed = torch.einsum('bijhw,bjchw->bichw', H_res, stream_r)  # (B, 4, 64, H, W)

        # --- Final: add the residual-mixed streams + the layer output ---
        #
        # new_stream = (H_res @ old_streams) + (H_post * layer_output)
        #            = mixed old streams     + gated new information
        #
        # At init this ≈ old_streams + small * layer_output, which is basically
        # a residual connection. Over training, the mixing becomes learned.
        new_stream = mixed + (self.output_gate * stream_update)
        return new_stream.reshape(B, nC, H, W)

class DeepSeekSparseAttention2D(nn.Module):
    def __init__(self, dim, num_main_heads=8, top_k=32):
        super().__init__()
        self.dim = dim
        self.top_k = top_k
        self.temperature = 1.3

        self.num_main_heads = num_main_heads
        self.main_head_dim = dim // num_main_heads
        assert dim % num_main_heads == 0

        # FIX 1: LayerNorm protects global image colors
        self.norm = nn.LayerNorm(dim)

        self.main_q_proj = nn.Linear(dim, dim, bias=False)
        self.main_k_proj = nn.Linear(dim, dim, bias=False)
        self.main_v_proj = nn.Linear(dim, dim, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)
        
        # FIX 2: Gamma Zero-Init (Starts the block as invisible)
        self.gamma = nn.Parameter(torch.zeros(1))

        self.scale = 1.0 / math.sqrt(self.main_head_dim)

        # FIX 3: The Garbage Can
#         self.sink_token = nn.Parameter(torch.randn(1, 1, dim) * 0.02)
        self.sink_token = nn.Parameter(torch.randn(1, 4, dim) * 0.02)

    def forward(self, x):
        B, C, H, W = x.shape
        N = H * W

        x_flat = x.flatten(2).transpose(1, 2)  
        x_flat = self.norm(x_flat)

        sink = self.sink_token.expand(B, -1, -1)
        x_flat = torch.cat([sink, x_flat], dim=1) 

        D = self.main_head_dim
        Heads = self.num_main_heads
        N_total = N + 4

        Q = self.main_q_proj(x_flat).view(B, N_total, Heads, D).permute(0, 2, 1, 3)
        K = self.main_k_proj(x_flat).view(B, N_total, Heads, D).permute(0, 2, 1, 3)
        V = self.main_v_proj(x_flat).view(B, N_total, Heads, D).permute(0, 2, 1, 3)

        raw_scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale  
        
        actual_k = min(self.top_k, N_total)
        _, top_indices = torch.topk(raw_scores, actual_k, dim=-1)
        
        mask = torch.zeros_like(raw_scores, dtype=torch.bool).scatter_(-1, top_indices, True)
        
        # FIX 4: The VIP Pass (Because we used topk, we MUST have this bouncer override)
        mask[:, :, :, :4] = True
        
        raw_scores = raw_scores.masked_fill(~mask, float('-inf'))

        attn = F.softmax(raw_scores / self.temperature, dim=-1)
        out = torch.matmul(attn, V)  

        # Slice off the sink token
        out = out[:, :, 4:, :] 

        out = out.permute(0, 2, 1, 3).reshape(B, N, C)
        out = self.out_proj(out)
        
        # THE FIX: We apply the gamma volume knob to the attention details,
        # reshape it to the image dimensions, and safely ADD the original input 'x'.
        return x + (self.gamma * out).transpose(1, 2).view(B, C, H, W)

# ============================================================================
#  mHC-ENHANCED MIDDLE BLOCK
# ============================================================================

class mHCMiddleBlock(nn.Module):
    """MiddleBlock with mHC wrapping the pre-conv, attention, and post-conv."""
    def __init__(self, channels=320, num_heads=8, expansion=4, sinkhorn_iters=5):
        super().__init__()
        self.channels = channels
        self.expansion = expansion

        # Stream enter/exit
        self.stream_init = nn.Conv2d(channels, channels * expansion, 1, bias=False)
        self.stream_out = nn.Conv2d(channels * expansion, channels, 1, bias=False)
        self._init_stream_projections(channels, expansion)

        # Layer 1: pre-conv
        self.mhc1 = mHCUnit(channels, expansion, sinkhorn_iters)
        self.pre = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, padding_mode='reflect', bias=False),
            GN(channels),
            nn.LeakyReLU(0.1, inplace=True),
        )

        # ==========================================
        # NEW: Layer 2: DeepSeek Sparse Attention
        # ==========================================
        self.mhc2 = mHCUnit(channels, expansion, sinkhorn_iters)
        # channels=320. 320 % 8 heads = 40 (perfect integer for head_dim)
        self.DSA_m_attn = DeepSeekSparseAttention2D(dim=320, num_main_heads=8, top_k=28)

        # Layer 3: post-conv + SE
        self.mhc3 = mHCUnit(channels, expansion, sinkhorn_iters)
        self.post = nn.Sequential(
            nn.Conv2d(channels, channels * 2, 3, padding=1, padding_mode='reflect', bias=False),
            GN(channels * 2),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(channels * 2, channels, 3, padding=1, padding_mode='reflect', bias=False),
            GN(channels),
        )
        self.se = SqueezeExcitation(channels, se_ratio=0.25)

    def _init_stream_projections(self, C, n):
        with torch.no_grad():
            w_in = torch.zeros(C * n, C, 1, 1)
            w_in[:C, :, 0, 0] = torch.eye(C)
            self.stream_init.weight.copy_(w_in)
            w_out = torch.zeros(C, C * n, 1, 1)
            w_out[:, :C, 0, 0] = torch.eye(C)
            self.stream_out.weight.copy_(w_out)

    def forward(self, x):
        stream = self.stream_init(x)
        
        stream = self.mhc1(stream, self.pre)
        stream = self.mhc2(stream, self.DSA_m_attn)
        stream = self.mhc3(stream, lambda z: self.se(self.post(z)))
        
        return self.stream_out(stream)

# ============================================================================
#  STANDARD DECODER BLOCK (for final_decoder which has no skip)
# ============================================================================

class DecoderBlock(nn.Module):
    """Original decoder block, used where mHC isn't needed (final stage)."""
    def __init__(self, in_ch, out_ch, se_ratio=0.25, upsample_scale=2, blur=False):
        super().__init__()
        self.contract = nn.Sequential(
            nn.Conv2d(in_ch, in_ch // 2, kernel_size=1, bias=False),
            GN(in_ch // 2), nn.LeakyReLU(0.1, inplace=True))
        self.upsample = PixelShuffle_ICNR(in_ch // 2, in_ch // 2,
                                           scale=upsample_scale, blur=blur)
        self.expand = nn.Sequential(
            nn.Conv2d(in_ch // 2, out_ch, kernel_size=1, bias=False),
            GN(out_ch), nn.LeakyReLU(0.1, inplace=True))
        self.post_expand_conv = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, 3, padding=1, padding_mode='reflect', bias=False),
            GN(out_ch), nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, padding_mode='reflect', bias=False),
            GN(out_ch), nn.LeakyReLU(0.1, inplace=True))
        self.se = SqueezeExcitation(out_ch, se_ratio)
        self.skip_conv = nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)

    def forward(self, x):
        skip = self.skip_conv(x)
        x = self.expand(self.upsample(self.contract(x)))
        res = x
        x = self.se(self.post_expand_conv(x)) + res
        skip = F.interpolate(skip, size=x.shape[2:], mode='bilinear',
                             align_corners=False)
        return x + skip


# ============================================================================
#  U-NET WITH mHC
# ============================================================================

class UNet(nn.Module):
    """
    EfficientNet-B1 encoder + mHC-enhanced decoder.

    mHC is applied to:
      - MiddleBlock (pre-conv, attention, post-conv)
      - Decoder blocks d1-d5 (refinement layers + skip fusion)

    mHC is NOT applied to:
      - Encoder (pretrained, leave unchanged)
      - RefinementBlock (too shallow to benefit)
      - final_decoder (takes RGB input concat, no encoder skip)
    """
    def __init__(self, num_classes=3, expansion=4):
        super().__init__()

        # Encoder (pretrained, frozen or fine-tuned)
        self.one_enc = EfficientNetB1SingleEncoder()
        
        self.refine_16_0 = SharedRefineBlock(320, num_heads=8, top_k=28)
        self.refine_16_iters = 8

        # Middle block with mHC
        self.middle = mHCMiddleBlock(channels=320, expansion=expansion)

        self.d0_norm = GN(320)

        self.refine_16_1 = SharedRefineBlock(320, num_heads=8, top_k=28)
        self.refine_16_iters = 8

        self.d1 = DecoderBlock(320*2, 180, blur=False)
        self.d1_norm = GN(180)

        self.refine_32 = SharedRefineBlock(180, num_heads=4, top_k=64)
        self.refine_32_iters = 6

        self.d2 = DecoderBlock(292, 128, blur=False)
        self.d2_norm = GN(128)

        self.refine_64 = SharedRefineBlock(128, num_heads=4, top_k=96)
        self.refine_64_iters = 4

        self.d3 = DecoderBlock(168, 72, blur=False)
        self.d3_norm = GN(72)

        # --- Decoder Stage 4: 128×128 → 256×256 (32ch, was 16) ---
        # in_ch = 48 + 24 (skip[3]) = 72
        self.d4 = DecoderBlock(96, 48, blur=False)
        self.d4_norm = GN(48)

        # --- Decoder Stage 5: 256×256 → 512×512 (32ch, was 16) ---
        # in_ch = 32 + 16 (skip[4]) = 48
        self.d5 = DecoderBlock(64, 42, blur=False)

        # --- Final 2× upsample to 1024×1024 ---
        self.refinement_0 = RefinementBlock(42, 42, 42)
        self.final_decoder_upsample = DecoderBlock(45, 32, blur=False)
        self.spatial_attn = SpatialAttention(kernel_size=7)
        self.refinement_1 = RefinementBlock(32, 12, 3)

    def forward(self, x):
        initial_x = x
        x, skips = self.one_enc(x.float())
        
        residual = x
        for _ in range(self.refine_16_iters):
            x = self.refine_16_0(x)
        x = x + residual
        
        x = self.middle(x.float())
        x = self.d0_norm(x)

        residual = x
        for _ in range(self.refine_16_iters):
            x = self.refine_16_1(x)
        x = x + residual

        # --- Stage 1: 16×16 → 32×32 ---
        x = self.d1(torch.cat([x, skips[0]], dim=1))
        x = self.d1_norm(x)

        residual = x
        # Shared refinement loop at 32×32 (6 passes, same weights)
        for _ in range(self.refine_32_iters):
            x = self.refine_32(x)
        x = x + residual

        # --- Stage 2: 32×32 → 64×64 ---
        x = self.d2(torch.cat([x, skips[1]], dim=1))
        x = self.d2_norm(x)

        residual = x
        # Shared refinement loop at 64×64 (4 passes, same weights)
        for _ in range(self.refine_64_iters):
            x = self.refine_64(x)
        x = x + residual

        # --- Stage 3-5: no DSA (too expensive at 128×128+) ---
        x = self.d3(torch.cat([x, skips[2]], dim=1))
        x = self.d3_norm(x)
        x = self.d4(torch.cat([x, skips[3]], dim=1))
        x = self.d4_norm(x)
        x = self.d5(torch.cat([x, skips[4]], dim=1))

        # --- Final upsample 512→1024 ---
        x = self.refinement_0(x)
        x = self.final_decoder_upsample(torch.cat([x, initial_x], dim=1))
        x = self.spatial_attn(x)
        x = self.refinement_1(x)
        return x


In [3]:
model = UNet(expansion=4).cuda()

C:\Users\ooo\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\modules\module.py:1082: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:35.)
  return self._apply(lambda t: t.cuda(device))


In [4]:
r = torch.rand(2, 3, 512, 512).cuda()
with torch.no_grad():
    m=model(r)

In [5]:
total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params:      {total_params:,}")
print(f"Trainable params: {trainable:,}")

Total params:      15,596,321
Trainable params: 15,596,321


In [53]:
next(im)

2

In [83]:
counter = next(im)
print(counter)
r = torch.rand(4, 3, counter, counter).cuda()
with torch.no_grad():
    m=model(r)


32


In [91]:
m.shape

torch.Size([4, 3, 64, 64])

In [6]:
# ============================================================================
#  DEBUG SAVE
# ============================================================================

def save_debug_triplet(model, y_folder, filenames, counter, epoch,
                       save_dir="save_image_folder", num_samples=2,
                       x_size=512, y_size=1024, pretrain=False):
    """
    Save X / Y_hat / Y triplets for visual inspection.
    In pretrain mode: uses clean small images matching training.
    In full mode:     uses degraded 512→1024 images.
    """
    os.makedirs(save_dir, exist_ok=True)
    model.eval()
    with torch.no_grad():
        # THE FIX: Sample without replacement to guarantee unique images per call
        sampled_fnames = random.sample(filenames, min(num_samples, len(filenames)))
        
        for i, fname in enumerate(sampled_fnames):
            img = Image.open(os.path.join(y_folder, fname)).convert("RGB")

            # Build X and Y at the same sizes the model was trained on
            y_img = TF.resize(img, (y_size, y_size), interpolation=InterpolationMode.BICUBIC)
            x_img = TF.resize(img, (x_size, x_size), interpolation=InterpolationMode.BICUBIC)

            x_t = transform_x(x_img).unsqueeze(0).cuda()
#             x_t = transform_clean(x_img).unsqueeze(0).cuda()
            x = transform_clean(x_img).unsqueeze(0).cuda()
            y_t = transform_clean(y_img).unsqueeze(0).cuda()

            with autocast('cuda'):
                y_hat = model(x_t)

            tag = f"{counter}_{epoch}_{i}_{uuid.uuid4().hex[:6]}_{os.path.splitext(fname)[0]}"
            for suffix, t in [("x", x), ("x_blur", x_t), ("yhat", y_hat), ("y", y_t)]:
                tensor_to_image(denormalize(t[0].detach())).save(
                    os.path.join(save_dir, f"{tag}_{suffix}.png"))
            print(f"  Saved: {tag}")
    model.train()

In [7]:
# model = UNet(expansion=4).cuda()
save_path = 'future_mhc_unet_weights.pth'

# # ---- MODEL ----
if os.path.exists(save_path):
    model.load_state_dict(torch.load(save_path, map_location='cuda:0'), strict=False)
    print(f"Loaded checkpoint: {save_path} (strict=False)")

Loaded checkpoint: future_mhc_unet_weights.pth (strict=False)


In [8]:
for n, (name, param) in enumerate(model.named_parameters()):
    if n < 294:
        param.requires_grad = False
    else:
        param.requires_grad = True
    print(n, f"Layer: {name} | Requires Grad: {param.requires_grad}")

0 Layer: one_enc.features.0.0.weight | Requires Grad: False
1 Layer: one_enc.features.0.1.weight | Requires Grad: False
2 Layer: one_enc.features.0.1.bias | Requires Grad: False
3 Layer: one_enc.features.1.0.block.0.0.weight | Requires Grad: False
4 Layer: one_enc.features.1.0.block.0.1.weight | Requires Grad: False
5 Layer: one_enc.features.1.0.block.0.1.bias | Requires Grad: False
6 Layer: one_enc.features.1.0.block.1.fc1.weight | Requires Grad: False
7 Layer: one_enc.features.1.0.block.1.fc1.bias | Requires Grad: False
8 Layer: one_enc.features.1.0.block.1.fc2.weight | Requires Grad: False
9 Layer: one_enc.features.1.0.block.1.fc2.bias | Requires Grad: False
10 Layer: one_enc.features.1.0.block.2.0.weight | Requires Grad: False
11 Layer: one_enc.features.1.0.block.2.1.weight | Requires Grad: False
12 Layer: one_enc.features.1.0.block.2.1.bias | Requires Grad: False
13 Layer: one_enc.features.1.1.block.0.0.weight | Requires Grad: False
14 Layer: one_enc.features.1.1.block.0.1.weight 

In [7]:
#For Linux with Triton installed only
# model = torch.compile(model)

In [39]:
disc = PatchDiscriminator(in_channels=3).cuda()


In [ ]:
r = torch.rand(2, 3, 512, 512).cuda()


In [32]:
disc(r).shape

torch.Size([2, 1, 62, 62])

In [62]:
from collections import OrderedDict
import torch
import torch.nn as nn
import torch.nn.functional as F



class PatchDiscriminator(nn.Module):
    """PatchGAN discriminator enhanced with SharedRefineBlocks.
    Downsamples to affordable resolutions first, then applies DSA + depthwise
    separable conv loops for powerful texture discrimination.
    Uses spectral normalization for stable GAN training."""
    def __init__(self, in_channels=3):
        super().__init__()

        def disc_block(in_c, out_c, stride=2, norm=True):
            layers = [nn.utils.spectral_norm(
                nn.Conv2d(in_c, out_c, kernel_size=4, stride=stride, padding=1, bias=not norm)
            )]
            if norm:
                layers.append(nn.InstanceNorm2d(out_c, affine=True))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return nn.Sequential(*layers)

        # Downsample path: 1024 → 512 → 256 → 128 → 64 → 32
        self.down1 = disc_block(in_channels, 64, stride=2, norm=False)  # 512
        self.down2 = disc_block(64, 128, stride=2)                       # 256
        self.down3 = disc_block(128, 256, stride=2)                      # 128
        self.down4 = disc_block(256, 256, stride=2)                      # 64
        self.down5 = disc_block(256, 512, stride=2)                      # 32

        # Final classification
        self.final = nn.Sequential(
            disc_block(512, 512, stride=1),                              # 31
            nn.utils.spectral_norm(
                nn.Conv2d(512, 1, kernel_size=4, stride=1, padding=1, padding_mode='reflect')    # 30
            ),
        )

    def forward(self, x):
        x = self.down1(x)
        x = self.down2(x)
        x = self.down3(x)
        x = self.down4(x)
        x = self.down5(x)

        return self.final(x)



class FeatureLoss(nn.Module):
    def __init__(self, weights, device='cuda:0'):
        super().__init__()
        from torchvision.models import efficientnet_b7, EfficientNet_B7_Weights

        self.backbone = efficientnet_b7(weights=EfficientNet_B7_Weights.DEFAULT).to(device).eval()
        for p in self.backbone.parameters():
            p.requires_grad = False

        self.layer_names = [
#             'features.1.1.stochastic_depth', # Shallow: High-frequency edges/noise
#             'features.3.3.stochastic_depth', # Mid: Local patterns/textures
#             'features.5.9.stochastic_depth', # Deep: Structure
#             'features.7.3.stochastic_depth', # Deep: Semantics
            
            'features.1.1',
            'features.3.3',
            'features.5.9',
            'features.7.3',
        ]
        self.features = OrderedDict()
        self.hooks = []
        for name, module in self.backbone.named_modules():
            if name in self.layer_names:
                self.hooks.append(module.register_forward_hook(self._make_hook(name)))
        self.weights = weights

    def _make_hook(self, name):
        def hook(module, inp, out):
            self.features[name] = out
        return hook

    @staticmethod
    def gram_matrix(x):
        b, c, h, w = x.size()
        x = x.view(b, c, -1)
        # Divide ONLY by h * w. This computes the spatial average of the feature products
        return torch.bmm(x, x.transpose(1, 2)) / (h * w)

    def charbonnier(self, pred, target, eps=1e-6):
        return torch.mean(torch.sqrt((pred.float() - target.float()) ** 2 + eps))

    def frequency_loss(self, pred, target, alpha=1.0):
        """
        Focal Frequency Loss — emphasizes hard-to-reconstruct frequencies.
        norm='ortho' prevents the loss from scaling with image resolution.
        """
        pred_fft = torch.fft.rfft2(pred.float(), norm='ortho')
        target_fft = torch.fft.rfft2(target.float(), norm='ortho')
        diff = torch.abs(pred_fft - target_fft)
        weight = diff.detach() ** alpha
        weight = weight / (weight.mean() + 1e-8)

        return (weight * diff).mean()

    def gradient_loss(self, pred, target):
        """Forces the model to match the physical sharpness of edges."""
        pred_dy = torch.abs(pred[:, :, 1:, :] - pred[:, :, :-1, :])
        pred_dx = torch.abs(pred[:, :, :, 1:] - pred[:, :, :, :-1])
        
        target_dy = torch.abs(target[:, :, 1:, :] - target[:, :, :-1, :])
        target_dx = torch.abs(target[:, :, :, 1:] - target[:, :, :, :-1])
        
        return F.l1_loss(pred_dy, target_dy) + F.l1_loss(pred_dx, target_dx)

    def forward(self, pred, target):
        pixel_loss = self.charbonnier(pred, target) * 0.00003
        freq_loss = self.frequency_loss(pred, target) * 0.02
        grad_loss = self.gradient_loss(pred, target) * 2

        self.features.clear()
        with torch.no_grad():
            self.backbone(target)
            tgt_feats = [self.features[n].clone() for n in self.layer_names]

        self.features.clear()
        self.backbone(pred)
        pred_feats = [self.features[n] for n in self.layer_names]

        feature_losses = [
            F.mse_loss(pf.float(), tf.float()) * w * 0.06
            for pf, tf, w in zip(pred_feats[:3], tgt_feats[:3], self.weights)
        ]

        with torch.amp.autocast('cuda', enabled=False):
            gram_losses = [
                F.mse_loss(self.gram_matrix(pf.float()),
                           self.gram_matrix(tf.float())) * w * 0.04
                for pf, tf, w in zip(pred_feats[:3], tgt_feats[:3], self.weights)
            ]

        total = sum(feature_losses) + sum(gram_losses) + freq_loss + grad_loss + pixel_loss
        return total, feature_losses, gram_losses, freq_loss, grad_loss, pixel_loss

    def __del__(self):
        for h in self.hooks:
            h.remove()

In [57]:
20*0.05

1.0

In [47]:
class LightDiscriminator(nn.Module):
    """Discriminator built with every efficient component available:
    - DepthwiseSeparableConv for cheap strided downsampling
    - SharedRefineBlock (DSA + DepthwiseSepConv) with shared-weight loops
    - mHC multi-stream processing at 16×16
    ~2M params, ~5M+ effective compute through weight sharing.
    Runs fast FP16 for convs, with localized FP32 for attention/sinkhorn to prevent NaNs."""

    def __init__(self, in_channels=3, base_ch=256, expansion=4):
        super().__init__()
        self.base_ch = base_ch
        self.expansion = expansion

        def sep_down(in_c, out_c):
            """Spectrally-normalized depthwise separable conv, stride=2."""
            return nn.Sequential(
                nn.Conv2d(in_c, in_c, 4, stride=2, padding=1, padding_mode='reflect', groups=in_c, bias=False),
                nn.utils.spectral_norm(
                    nn.Conv2d(in_c, out_c, 1, bias=False)),
                nn.InstanceNorm2d(out_c, affine=True),
                nn.LeakyReLU(0.2, inplace=True),
            )

        # === STAGE 1: Dynamic Downsampling ===
        # First, map RGB to base channels without shrinking the spatial size
        self.proj_in = nn.Sequential(
            nn.utils.spectral_norm(
                nn.Conv2d(in_channels, base_ch, 3, padding=1, padding_mode='reflect', bias=False)
            ),
            nn.LeakyReLU(0.2, inplace=True)
        )
        
        # A pool of downsample blocks. 6 blocks allows up to 2048x2048 input.
        self.dynamic_downs = nn.ModuleList([sep_down(base_ch, base_ch) for _ in range(6)])

        # === STAGE 2: Shared DSA refinement at 32×32 ===
        self.refine_32 = SharedRefineBlock(base_ch, num_heads=4, top_k=32)
        self.refine_32_iters = 4

        # === STAGE 3: Down to 16×16 ===
        self.down_to_16 = sep_down(base_ch, base_ch)

        # === STAGE 4: Shared DSA refinement at 16×16 ===
        self.refine_16 = SharedRefineBlock(base_ch, num_heads=4, top_k=28)
        self.refine_16_iters = 6

        # === STAGE 5: mHC multi-stream processing at 16×16 ===
        self.mhc_stream_init = nn.Conv2d(base_ch, base_ch * expansion, 1, bias=False)
        self.mhc_stream_out = nn.Conv2d(base_ch * expansion, base_ch, 1, bias=False)
        self._init_mhc_streams(base_ch, expansion)

        self.mhc1 = mHCUnit(base_ch, expansion=expansion, sinkhorn_iters=5)
        self.mhc1_layer = DepthwiseSeparableConv(base_ch)

        self.mhc2 = mHCUnit(base_ch, expansion=expansion, sinkhorn_iters=5)
        self.mhc2_layer = DepthwiseSeparableConv(base_ch)

        # === STAGE 6: Final classification ===
        self.down_to_8 = sep_down(base_ch, base_ch)   # → 8×8
        
        # padding=0 requires no padding mode
        self.final = nn.utils.spectral_norm(
            nn.Conv2d(base_ch, 1, kernel_size=4, stride=1, padding=0)  # → 5×5 output grid
        )

    def _init_mhc_streams(self, C, n):
        """Initialize stream projections to near-identity (stream[0] = main signal)."""
        with torch.no_grad():
            w_in = torch.zeros(C * n, C, 1, 1)
            w_in[:C, :, 0, 0] = torch.eye(C)
            self.mhc_stream_init.weight.copy_(w_in)
            w_out = torch.zeros(C, C * n, 1, 1)
            w_out[:, :C, 0, 0] = torch.eye(C)
            self.mhc_stream_out.weight.copy_(w_out)

    def forward(self, x):
        orig_dtype = x.dtype  

        # ==============================================================
        # FAST CONV STAGE (Runs in FP16 if autocast is active)
        # ==============================================================
        # Map to 256 channels
        x = self.proj_in(x)
        
        # Dynamically loop through downsample blocks until size hits 32x32
        for down_block in self.dynamic_downs:
            if x.shape[2] <= 32:
                break
            x = down_block(x)

        # ==============================================================
        # ATTENTION STAGE (Forced to FP32 to prevent Softmax NaNs)
        # ==============================================================
        with torch.autocast('cuda', enabled=False):
            x_f32 = x.float()
            residual = x_f32
            for _ in range(self.refine_32_iters):
                x_f32 = self.refine_32(x_f32)
            x = (x_f32 + residual).to(orig_dtype)  # Cast back to fp16

        # ==============================================================
        # FAST CONV STAGE (Runs in FP16)
        # ==============================================================
        x = self.down_to_16(x)

        # ==============================================================
        # ATTENTION & SINKHORN STAGE (Forced to FP32)
        # ==============================================================
        with torch.autocast('cuda', enabled=False):
            x_f32 = x.float()
            
            # Shared DSA at 16x16
            residual = x_f32
            for _ in range(self.refine_16_iters):
                x_f32 = self.refine_16(x_f32)
            x_f32 = x_f32 + residual

            # mHC Streams
            stream = self.mhc_stream_init(x_f32)
            stream = self.mhc1(stream, self.mhc1_layer)
            stream = self.mhc2(stream, self.mhc2_layer)
            x = self.mhc_stream_out(stream).to(orig_dtype) # Cast back to fp16

        # ==============================================================
        # FINAL CLASSIFICATION (Runs in FP16)
        # ==============================================================
        x = self.down_to_8(x)
        return self.final(x)

In [45]:
r = torch.rand(2, 3, 256, 256).cuda()


In [49]:
with torch.no_grad():
    m=disc(r)

In [50]:
m.shape

torch.Size([2, 1, 5, 5])

In [48]:
disc = LightDiscriminator(in_channels=3).cuda()


In [23]:
im = iter([z for z in range(2, 300)])


In [40]:
counter = next(im)
print(counter)
r = torch.rand(4, 3, counter, counter).cuda()
with torch.no_grad():
    m=disc(r)


18


ValueError: Expected more than 1 spatial element when training, got input size torch.Size([4, 256, 1, 1])

In [ ]:
import csv
import os
import time
import random
import numpy as np
import torch
import torch.nn.functional as F
from torch.amp import GradScaler
import matplotlib.pyplot as plt

# ============================================================================
#  ADA: DIFFERENTIABLE AUGMENTATION
# ============================================================================
def diff_augment(x, p=0.0):
    """Differentiable augmentation: color + translation + cutout.
    All ops are differentiable — gradients flow through to the generator."""
    if p <= 0.0:
        return x
    
    B, C, H, W = x.shape
    
    # 1. Color: random brightness + contrast shift
    if torch.rand(1).item() < p:
        contrast = 1.0 + (torch.rand(B, 1, 1, 1, device=x.device) - 0.5) * 0.4
        brightness = (torch.rand(B, 1, 1, 1, device=x.device) - 0.5) * 0.2
        x = x * contrast + brightness
    
    # 2. Translation: pad with reflection, random crop back to original size
    #    This is the STRONGEST augment — forces disc to learn textures, not positions
    if torch.rand(1).item() < p:
        shift = max(1, int(H * 0.125))  # 12.5% = 128px for 1024×1024
        x = F.pad(x, [shift] * 4, mode='reflect')
        dy = torch.randint(0, 2 * shift + 1, (1,)).item()
        dx = torch.randint(0, 2 * shift + 1, (1,)).item()
        x = x[:, :, dy:dy + H, dx:dx + W]
    
    # 3. Cutout: erase a random 25% rectangle — removes spatial shortcuts
    if torch.rand(1).item() < p:
        cut_h = int(H * 0.25)
        cut_w = int(W * 0.25)
        cy = torch.randint(cut_h // 2, H - cut_h // 2, (1,)).item()
        cx = torch.randint(cut_w // 2, W - cut_w // 2, (1,)).item()
        mask = torch.ones(1, 1, H, W, device=x.device)
        mask[:, :, cy - cut_h//2 : cy + cut_h//2, cx - cut_w//2 : cx + cut_w//2] = 0
        x = x * mask
    
    return x


# time.sleep(60*60*8)
# ---- CONFIG (flip this to switch between pretrain and full training) ----
PRETRAIN_MODE = True          # True = clean 64→64, False = degraded 128→1024
USE_GAN = True                # True = Train with Discriminator, False = Bypass GAN entirely
y_folder = "E:/close_alexa/3y_1024_1024/"
batch_size = 1
accumulation_steps = 32  # Effective batch size = 1 * 32 = 32
lr = 2e-5
num_epochs = 1 if PRETRAIN_MODE else 40

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params:      {total_params:,}")
print(f"Trainable params: {trainable:,}")

# ---- LOSS / OPTIMIZER / DISCRIMINATOR ----
feat_loss = FeatureLoss(weights=[0.05, 0.07, 0.12, 0.8]).cuda()

# Generator optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scaler = GradScaler("cuda")

# Discriminator setup
disc = LightDiscriminator(in_channels=3).cuda()
disc_save_path = 'light_disc_weights.pth'

if os.path.exists(disc_save_path):
    disc.load_state_dict(torch.load(disc_save_path, map_location='cuda:0'), strict=False)
    print(f"Loaded discriminator checkpoint: {disc_save_path}")

disc_optimizer = torch.optim.Adam(disc.parameters(), lr=4e-5, betas=(0.5, 0.999))
disc_scaler = GradScaler("cuda")
adv_weight = 0.01  # Adversarial balance weight

# --- ADA INITIALIZATION ---
ada_p = 0.1        # Starting probability of augmentation
ada_target = 0.20    # The ideal D_loss equilibrium point
ada_step = 0.01      # How much to adjust p per accumulation step

disc_params = sum(p.numel() for p in disc.parameters())
print(f"Discriminator params: {disc_params:,}")

# ---- DATA ----
all_filenames = get_filenames_from_y(y_folder)
train_filenames, val_filenames = split_filenames(all_filenames)

print(f"Mode: {'PRETRAIN' if PRETRAIN_MODE else 'FULL'} | GAN: {'ON' if USE_GAN else 'OFF'} | "
      f"{len(train_filenames)} train, {len(val_filenames)} val | bs={batch_size}")

factor = 6

# ---- LOGGING SETUP ----
csv_filename = "training_loss_log.csv"
chart_filename = "loss_chart.png"

if not os.path.exists(csv_filename):
    with open(csv_filename, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "step", "train_loss", "val_loss", "ada_p"])

def save_loss_chart(csv_file, out_file):
    if not os.path.exists(csv_file):
        return
    steps, t_loss, v_loss = [], [], []
    with open(csv_file, mode='r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            steps.append(int(row["step"]))
            t_loss.append(float(row["train_loss"]))
            v_loss.append(float(row["val_loss"]))
    
    plt.figure(figsize=(10, 6))
    plt.plot(steps, t_loss, label='Train Loss', alpha=0.8)
    plt.plot(steps, v_loss, label='Validation Loss', alpha=0.8)
    plt.xlabel('Steps')
    plt.ylabel('Loss')
    plt.title('Training & Validation Loss Over Time')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.savefig(out_file)
    plt.close()

# ---- TRAINING ----
max_images = 55555555_000          
images_seen = 0              
global_step = 0              

for epoch in range(1, num_epochs + 1):
    print(f"\n{'='*60}\nEpoch {epoch}/{num_epochs}\n{'='*60}")
    random.shuffle(train_filenames)
    model.train()
    disc.train()
    counter = 0
    
    if epoch % 500 == 0:
        factor += 1

    # Buffers
    loss_buffer = []
    feature_losses_buffer = []
    gram_losses_buffer = []
    pixel_loss_buffer = []
    freq_loss_buffer = []
    grad_loss_buffer = []
    d_loss_buffer = []
    g_adv_loss_buffer = []
    ada_p_buffer = []

    if PRETRAIN_MODE:
        train_loader = load_pretrain_images(y_folder, train_filenames, batch_size,
                                            x_size=32*factor, y_size=64*factor, augment=True)
    else:
        train_loader = load_images_from_filenames(y_folder, train_filenames, batch_size, augment=True)

    optimizer.zero_grad()
    disc_optimizer.zero_grad()
    start_time = time.time()
    
    d_loss_val = 0.0
    g_adv_loss_val = 0.0

    for x_batch, y_batch in train_loader:
        counter += 1
        global_step += 1
        
        if counter == 100:
            elapsed = time.time() - start_time
            print(f"\n⏱️ [SPEED TEST] 100 steps took: {elapsed:.2f} seconds\n")
        
        images_seen += x_batch.size(0)

        if images_seen >= max_images:
            print(f"Reached {max_images} images. Stopping training early.")
            break

        # ==============================================================
        # 1. GENERATOR FORWARD & BACKWARD
        # ==============================================================
        # Freeze Discriminator to prevent double-gradient computation
        if USE_GAN:
            for p in disc.parameters():
                p.requires_grad = False

        with torch.autocast('cuda'):
            outputs = model(x_batch)
            loss, feature_losses, gram_losses, freq_loss, grad_loss, pixel_loss = feat_loss(outputs, y_batch)

            if USE_GAN:
                # ADA: Apply identical augmentations before passing to D
                aug_outputs = diff_augment(outputs, p=ada_p)
                
                d_fake_for_g = disc(aug_outputs)
                
                with torch.no_grad():
                    d_real_for_g = disc(diff_augment(y_batch, p=ada_p))
                    
                g_adv_loss = F.binary_cross_entropy_with_logits(
                    d_fake_for_g - d_real_for_g.mean(0, keepdim=True),
                    torch.ones_like(d_fake_for_g)) * adv_weight
            else:
                g_adv_loss = torch.tensor(0.0, device='cuda')

            total_loss = loss + g_adv_loss
            scaled_g_loss = total_loss / accumulation_steps

        if torch.isnan(scaled_g_loss) or torch.isinf(scaled_g_loss):
            print(f"  ⚠ BAD LOSS at step {counter}, skipping")
            continue

        scaler.scale(scaled_g_loss).backward()

        # ==============================================================
        # 2. DISCRIMINATOR FORWARD & BACKWARD
        # ==============================================================
        if USE_GAN:
            # Unfreeze Discriminator for its own update
            for p in disc.parameters():
                p.requires_grad = True

            with torch.autocast('cuda'):
                # .detach() prevents gradients from flowing back into the Generator here
                fake_detached = outputs.detach()
                
                # ADA: Apply identical augmentations
                aug_y = diff_augment(y_batch, p=ada_p)
                aug_fake = diff_augment(fake_detached, p=ada_p)

                d_real = disc(aug_y)
                d_fake = disc(aug_fake)

                d_loss_real = F.binary_cross_entropy_with_logits(
                    d_real - d_fake.mean(0, keepdim=True), torch.ones_like(d_real))
                d_loss_fake = F.binary_cross_entropy_with_logits(
                    d_fake - d_real.mean(0, keepdim=True), torch.zeros_like(d_fake))
                
                d_loss = (d_loss_real + d_loss_fake) / 2
                scaled_d_loss = d_loss / accumulation_steps
                
            disc_scaler.scale(scaled_d_loss).backward()

            d_loss_val = d_loss.item()
        
        g_adv_loss_val = g_adv_loss.item() if USE_GAN else 0.0

        # ==============================================================
        # 3. OPTIMIZER STEPS (Executed every accumulation_steps)
        # ==============================================================
        if counter % accumulation_steps == 0:
            # --- Generator Step ---
            scaler.unscale_(optimizer)
            total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            if total_norm > 60:
                print(f"  ⚠ Large G gradient norm: {total_norm:.2f}")
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

            if USE_GAN:
                # --- Discriminator Step ---
                disc_scaler.unscale_(disc_optimizer)
                torch.nn.utils.clip_grad_norm_(disc.parameters(), max_norm=5.0)
                disc_scaler.step(disc_optimizer)
                disc_scaler.update()
                disc_optimizer.zero_grad()

                # --- ADA Update Step ---
                # If D is winning (loss < target), blur its vision
                if d_loss_val < ada_target:
                    ada_p = min(1.0, ada_p + ada_step)
                else:
                    ada_p = max(0.0, ada_p - ada_step)

        # Buffer appends
        loss_buffer.append(total_loss.item())
        feature_losses_buffer.append([z.item() for z in feature_losses])
        gram_losses_buffer.append([z.item() for z in gram_losses])
        freq_loss_buffer.append(freq_loss.item())
        grad_loss_buffer.append(grad_loss.item())
        pixel_loss_buffer.append(pixel_loss.item())
        d_loss_buffer.append(d_loss_val)
        g_adv_loss_buffer.append(g_adv_loss_val)
        ada_p_buffer.append(ada_p)

        if counter % 25 == 0:
            loss_mean = np.mean(loss_buffer)
            feat_mean = np.mean(feature_losses_buffer, axis=0)
            gram_mean = np.mean(gram_losses_buffer, axis=0)
            freq_mean = np.mean(freq_loss_buffer)
            grad_mean = np.mean(grad_loss_buffer)
            pixel_mean = np.mean(pixel_loss_buffer)
            d_loss_mean = np.mean(d_loss_buffer)
            g_adv_mean = np.mean(g_adv_loss_buffer)
            ada_p_mean = np.mean(ada_p_buffer)

            print(f'Epoch [{epoch}/{num_epochs}], Step [{counter}], '
                  f'Training Loss: {loss_mean:.5f}')
            print(f'  Freq: {freq_mean:.4f} | Grad: {grad_mean:.4f} | Pixel: {pixel_mean:.4f}')
            if USE_GAN:
                print(f'  D_loss: {d_loss_mean:.4f} | G_adv: {g_adv_mean:.4f} | ADA_p: {ada_p_mean:.3f}')
            print(f'  Features: {np.round(feat_mean, 4)}')
            print(f'  Gram:     {np.round(gram_mean, 4)}')

            model.eval()
            if USE_GAN:
                disc.eval()
                
            with torch.no_grad(), torch.autocast('cuda'):
                if PRETRAIN_MODE:
                    val_iter = load_pretrain_images(y_folder, val_filenames, batch_size,
                                                    x_size=32*factor, y_size=64*factor)
                else:
                    val_iter = load_images_from_filenames(y_folder, val_filenames, batch_size,
                                                          augment=False)
                x_val, y_val = next(iter(val_iter))
                val_out = model(x_val)
                val_loss = feat_loss(val_out, y_val)[0]
            print(f'  Validation Loss: {val_loss.item():.5f}')
            
            model.train()
            if USE_GAN:
                disc.train()

            # --- WRITE TO CSV ---
            with open(csv_filename, mode='a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow([epoch, global_step, loss_mean, val_loss.item(), ada_p])

        if counter % 500 == 0:
            torch.save(model.state_dict(), save_path)
            if USE_GAN:
                torch.save(disc.state_dict(), disc_save_path)
                print(f"  Model + Discriminator saved")
            else:
                print(f"  Model saved")

        if counter % 180 == 0:
            save_loss_chart(csv_filename, chart_filename)
            print(f"  Chart saved to {chart_filename}")
            
            print("  Saving debug triplet...")
        
            if PRETRAIN_MODE:
                save_debug_triplet(model, y_folder, train_filenames, counter, epoch,
                                   x_size=32*factor, y_size=64*factor, pretrain=True)
            else:
                save_debug_triplet(model, y_folder, train_filenames, counter, epoch)

    if images_seen >= max_images:
        break

# Final save
torch.save(model.state_dict(), save_path)
if USE_GAN:
    torch.save(disc.state_dict(), disc_save_path)
print(f"\nTraining complete. Model {'+ Discriminator ' if USE_GAN else ''}saved.")

Total params:      15,596,321
Trainable params: 9,495,937
Loaded discriminator checkpoint: light_disc_weights.pth
Discriminator params: 1,948,477
Mode: PRETRAIN | GAN: ON | 263643 train, 10986 val | bs=1

Epoch 1/1
Epoch [1/1], Step [25], Training Loss: 0.51800
  Freq: 0.1400 | Grad: 0.0933 | Pixel: 0.0000
  D_loss: 0.1270 | G_adv: 0.0330 | ADA_p: 0.100
  Features: [0.0084 0.0497 0.043 ]
  Gram:     [0.0636 0.0464 0.0405]
  Validation Loss: 0.15344
Epoch [1/1], Step [50], Training Loss: 0.51382
  Freq: 0.1129 | Grad: 0.1070 | Pixel: 0.0000
  D_loss: 0.2283 | G_adv: 0.0308 | ADA_p: 0.104
  Features: [0.0083 0.0521 0.0468]
  Gram:     [0.0584 0.0496 0.0478]
  Validation Loss: 0.37702
Epoch [1/1], Step [75], Training Loss: 0.46676
  Freq: 0.0901 | Grad: 0.1019 | Pixel: 0.0000
  D_loss: 0.1999 | G_adv: 0.0327 | ADA_p: 0.107
  Features: [0.0074 0.0511 0.0448]
  Gram:     [0.0458 0.0488 0.0442]
  Validation Loss: 0.21781

⏱️ [SPEED TEST] 100 steps took: 75.25 seconds

Epoch [1/1], Step [100]

  Validation Loss: 0.28789
Epoch [1/1], Step [775], Training Loss: 0.39333
  Freq: 0.0478 | Grad: 0.1045 | Pixel: 0.0000
  D_loss: 0.2336 | G_adv: 0.0332 | ADA_p: 0.181
  Features: [0.0054 0.0505 0.0442]
  Gram:     [0.0222 0.0451 0.0405]
  Validation Loss: 0.21335
Epoch [1/1], Step [800], Training Loss: 0.39392
  Freq: 0.0487 | Grad: 0.1040 | Pixel: 0.0000
  D_loss: 0.2359 | G_adv: 0.0330 | ADA_p: 0.183
  Features: [0.0054 0.0503 0.044 ]
  Gram:     [0.023  0.045  0.0405]
  Validation Loss: 0.18560
Epoch [1/1], Step [825], Training Loss: 0.39619
  Freq: 0.0489 | Grad: 0.1049 | Pixel: 0.0000
  D_loss: 0.2367 | G_adv: 0.0327 | ADA_p: 0.185
  Features: [0.0055 0.0504 0.0442]
  Gram:     [0.0234 0.0452 0.0409]
  Validation Loss: 0.24521
Epoch [1/1], Step [850], Training Loss: 0.39769
  Freq: 0.0492 | Grad: 0.1046 | Pixel: 0.0000
  D_loss: 0.2347 | G_adv: 0.0326 | ADA_p: 0.187
  Features: [0.0055 0.0505 0.0443]
  Gram:     [0.0237 0.046  0.0414]
  Validation Loss: 0.29994
Epoch [1/1], Step

  Validation Loss: 0.27153
Epoch [1/1], Step [1575], Training Loss: 0.39849
  Freq: 0.0512 | Grad: 0.1042 | Pixel: 0.0000
  D_loss: 0.2565 | G_adv: 0.0321 | ADA_p: 0.223
  Features: [0.0054 0.0506 0.0443]
  Gram:     [0.0233 0.0466 0.0409]
  Validation Loss: 0.29186
Epoch [1/1], Step [1600], Training Loss: 0.39848
  Freq: 0.0519 | Grad: 0.1035 | Pixel: 0.0000
  D_loss: 0.2586 | G_adv: 0.0321 | ADA_p: 0.224
  Features: [0.0054 0.0505 0.0443]
  Gram:     [0.0234 0.0465 0.0409]
  Validation Loss: 0.20934
  Chart saved to loss_chart.png
  Saving debug triplet...
  Saved: 1620_1_0_b7e58c_Dolby Vision 12K HDR 240fps  Experience the Unbelievable Nature   8K Earth_frame_2406s
  Saved: 1620_1_1_d1a799_frame_075267
Epoch [1/1], Step [1625], Training Loss: 0.39784
  Freq: 0.0517 | Grad: 0.1034 | Pixel: 0.0000
  D_loss: 0.2593 | G_adv: 0.0320 | ADA_p: 0.225
  Features: [0.0054 0.0505 0.0442]
  Gram:     [0.0233 0.0465 0.0409]
  Validation Loss: 0.32055
Epoch [1/1], Step [1650], Training Loss: 0.39

In [66]:
torch.save(model.state_dict(), save_path)
torch.save(disc.state_dict(), disc_save_path)